# ⚙️ Notebook 02 — Data Preprocessing

**Project:** ASD Detection in Children using Machine Learning  

---

## 🎯 Objectives

In this notebook we will:
1. Clean the raw dataset (remove noise, fix types)
2. Handle missing values
3. Encode categorical and binary features
4. Scale numeric features
5. Perform feature selection / importance screening
6. Save the cleaned dataset for modeling

---

## 📌 Why Preprocessing Matters

Raw data is rarely model-ready. ML algorithms:
- Cannot handle `NaN` values directly (most sklearn estimators)
- May be biased by large-scale numeric features (KNN, SVM, LR)
- Cannot interpret string categories like `'yes'`, `'m'`, `'White-European'`

A clean, well-prepared dataset is the **single most important factor** in model quality.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
import os

pd.set_option('display.max_columns', 50)

plt.rcParams.update({
    'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'text.color': '#e2eaf5', 'figure.dpi': 110,
})

print('✅ Imports complete')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/data_csv.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 2. Data Cleaning

### 2a. Strip whitespace from column names and string values

In [ ]:
# Strip column name whitespace
df.columns = df.columns.str.strip()

# Strip object column values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

print('✅ Whitespace stripped')
print('Columns:', list(df.columns))

### 2b. Encode binary columns (yes/no → 1/0)

In [ ]:
yn_map = {'yes': 1, 'no': 0, '1': 1, '0': 0, 'true': 1, 'false': 0}

binary_cols = [
    'Jaundice', 'Family_mem_with_ASD',
    'Speech Delay/Language Disorder', 'Learning disorder',
    'Genetic_Disorders', 'Depression',
    'Global developmental delay/intellectual disability',
    'Social/Behavioural Issues', 'Anxiety disorder'
]

for col in binary_cols:
    if col in df.columns:
        before = df[col].unique()
        df[col] = df[col].str.lower().map(yn_map).fillna(0).astype(int)
        print(f'  {col:<55} {str(before):<30} → {df[col].unique()}')

print('\n✅ Binary columns encoded')

### 2c. Encode Sex (m/f → 1/0)

In [ ]:
if 'Sex' in df.columns:
    df['Sex'] = df['Sex'].str.lower().map({'m':1,'male':1,'f':0,'female':0,'1':1,'0':0}).fillna(0).astype(int)
    print(f'Sex values: {df["Sex"].value_counts().to_dict()}')

### 2d. Convert Age to numeric

In [ ]:
if 'Age_Mons' in df.columns:
    df['Age_Mons'] = pd.to_numeric(df['Age_Mons'], errors='coerce')
    n_missing = df['Age_Mons'].isna().sum()
    if n_missing > 0:
        df['Age_Mons'].fillna(df['Age_Mons'].median(), inplace=True)
        print(f'Filled {n_missing} missing Age_Mons with median.')
    print(f'Age_Mons range: {df["Age_Mons"].min():.0f} – {df["Age_Mons"].max():.0f} months')

### 2e. Encode Target (YES/NO → 1/0)

In [ ]:
target_col = 'Class/ASD'
df[target_col] = df[target_col].str.upper().map({'YES':1,'NO':0}).fillna(0).astype(int)
print(f'Target distribution: {df[target_col].value_counts().to_dict()}')

## 3. Handle Categorical Feature: Ethnicity

We use **One-Hot Encoding** for Ethnicity because it has no ordinal relationship. This creates a binary column per category.

In [ ]:
if 'Ethnicity' in df.columns:
    df['Ethnicity'] = df['Ethnicity'].fillna('Others').replace({'nan':'Others', '':'Others'})
    print('Ethnicity value counts:')
    print(df['Ethnicity'].value_counts())

    # One-hot encode
    eth_dummies = pd.get_dummies(df['Ethnicity'], prefix='eth', drop_first=False)
    df = pd.concat([df.drop('Ethnicity', axis=1), eth_dummies], axis=1)
    print(f'\n✅ Added {len(eth_dummies.columns)} ethnicity dummy columns')

## 4. Feature Scaling

We apply **StandardScaler** (zero mean, unit variance) to numeric features. This is critical for:
- Logistic Regression
- KNN (distance-based)
- SVM

Tree-based models (Decision Tree, Random Forest) are scale-invariant, but we scale anyway for pipeline consistency.

In [ ]:
numeric_features = ['Age_Mons', 'Qchat-10-Score'] + \
                   (['Childhood Autism Rating Scale (CARS)'] if 'Childhood Autism Rating Scale (CARS)' in df.columns else [])

numeric_features = [c for c in numeric_features if c in df.columns]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_features] = scaler.fit_transform(df[numeric_features])

print('Before scaling:')
print(df[numeric_features].describe().T[['mean','std','min','max']].round(2))
print('\nAfter scaling:')
print(df_scaled[numeric_features].describe().T[['mean','std','min','max']].round(4))

## 5. Feature Selection

Using **mutual information** to rank features by their statistical relationship with the target.

In [ ]:
feature_cols = [c for c in df_scaled.columns if c != target_col]
X = df_scaled[feature_cols].fillna(0)
y = df_scaled[target_col]

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(mi_df)))
ax.barh(mi_df.index[::-1], mi_df.values[::-1], color=colors[::-1], edgecolor='#0f172a')
ax.set_title('Feature Importance via Mutual Information (Top 20)', fontsize=12)
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print('\nTop 10 features by mutual information:')
print(mi_df.head(10))

## 6. Final Cleaned Dataset

In [ ]:
print(f'Final shape: {df_scaled.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Target balance: {y.value_counts().to_dict()}')
print('\nData types:')
print(df_scaled.dtypes.value_counts())

# Save cleaned data
out_path = '../data/data_cleaned.csv'
df_scaled.to_csv(out_path, index=False)
print(f'\n✅ Cleaned dataset saved → {out_path}')

## 7. Preprocessing Summary

| Step | Action | Reason |
|---|---|---|
| Column cleaning | Strip whitespace | Avoid hidden key mismatch bugs |
| Binary encoding | yes/no → 1/0, m/f → 1/0 | sklearn requires numeric input |
| Target encoding | YES/NO → 1/0 | Binary classification |
| Missing values | Median fill (numeric), mode fill (categorical) | Preserve sample size |
| OHE (Ethnicity) | `get_dummies` | No ordinal assumption |
| Scaling | StandardScaler | Required for LR, KNN, SVM |
| Feature selection | Mutual info ranking | Understand predictive power |

---

## ➡️ Next: Notebook 03 — Model Training